# Perfilado de `interim` y generación de manifiesto de validación

Este notebook:

1. Lee los parquets ya procesados en **`data/interim/`** (post-ETL), para las fuentes configuradas en `etl.sources`.
2. Toma los **últimos N meses** con datos disponibles (por defecto **12**).
3. **Perfilá** cada columna: tipo observado, % nulos, cardinalidad, valores distintos si la cardinalidad es baja (candidatos a *enum*), min/max en numéricos y fechas.
4. Escribe un borrador **`config/raw_manifest.generated.yaml`** para revisión humana y uso futuro en un validador (`validate_raw.py` u otro).

No perfila columnas vacías (100 % nulos) ni **`localidad1`** (columna descartada en el ETL).

**Importante:** el manifiesto inferido **no** reemplaza el criterio de negocio: conviene revisar enums y umbrales antes de congelar una versión oficial (por ejemplo copiando a `config/raw_manifest.yaml`).

**Requisito:** existan particiones `data/interim/<fuente>/year=AAAA/month=MM/*.parquet`.

In [9]:
import os
import re
import sys
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from src.config import load_config, get_paths

## Parámetros

- **`N_MONTHS`**: cuántos meses hacia atrás considerar (recomendado: **12**).
- **`MAX_CARDINALITY_FOR_ENUM`**: si una columna tiene a lo sumo esta cantidad de valores distintos en el período, se listan como `allowed_values` en el manifiesto.
- **`OUTPUT_MANIFEST`**: ruta del YAML generado (no pisa un manifiesto oficial si usás otro nombre).

In [10]:
N_MONTHS = 12
MAX_CARDINALITY_FOR_ENUM = 50
MAX_ENUM_VALUES_LISTED = 80  # tope de valores listados por columna

OUTPUT_MANIFEST = os.path.join(module_path, "config", "raw_manifest.generated.yaml")

cfg = load_config()
paths = get_paths(cfg)
INTERIM_DIR = paths["interim"]
SOURCES = cfg.get("etl", {}).get("sources", ["inspecciones", "consumo"])

print("INTERIM_DIR:", INTERIM_DIR)
print("SOURCES:", SOURCES)
print("N_MONTHS:", N_MONTHS)

INTERIM_DIR: d:\2024\BID\Aguas\Empresa-EPM\proyecto\ar-epm_poc\data/interim
SOURCES: ['inspecciones', 'consumo']
N_MONTHS: 12


## Descubrir particiones `year=` / `month=`

In [11]:
_PART_RE = re.compile(r"year=(\d{4})/month=(\d{2})")


def list_partitions(interim_dir: str, source: str):
    """Lista (year, month) que tienen parquet para la fuente."""
    base = os.path.join(interim_dir, source)
    pattern = os.path.join(base, "year=*", "month=*", f"{source}.parquet")
    out = set()
    for fpath in glob.glob(pattern):
        m = _PART_RE.search(fpath.replace("\\", "/"))
        if m:
            out.add((int(m.group(1)), int(m.group(2))))
    return sorted(out)


def take_last_n_months(partitions: list, n: int) -> list:
    """partitions: lista ordenada de (y, m)."""
    if not partitions:
        return []
    return partitions[-n:]


for s in SOURCES:
    parts = list_partitions(INTERIM_DIR, s)
    print(f"{s}: {len(parts)} particiones; últimas 5: {parts[-5:]}")

inspecciones: 23 particiones; últimas 5: [(2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12)]
consumo: 42 particiones; últimas 5: [(2025, 2), (2025, 3), (2025, 4), (2025, 5), (2025, 6)]


## Cargar y concatenar los últimos N meses por fuente

In [12]:
def load_source_months(interim_dir: str, source: str, months: list) -> pd.DataFrame:
    frames = []
    for y, m in months:
        fp = os.path.join(
            interim_dir, source, f"year={y}", f"month={m:02d}", f"{source}.parquet"
        )
        if not os.path.isfile(fp):
            print(f"  [skip] no existe {fp}")
            continue
        df = pd.read_parquet(fp)
        df["_manifest_year"] = y
        df["_manifest_month"] = m
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


data_by_source = {}
for s in SOURCES:
    parts = list_partitions(INTERIM_DIR, s)
    chosen = take_last_n_months(parts, N_MONTHS)
    print(f"{s}: usando {len(chosen)} meses {chosen[:2]}...{chosen[-2:] if len(chosen) > 2 else chosen}")
    data_by_source[s] = load_source_months(INTERIM_DIR, s, chosen)
    print(f"  -> filas concatenadas: {len(data_by_source[s])}")

inspecciones: usando 12 meses [(2024, 1), (2024, 2)]...[(2024, 11), (2024, 12)]
  -> filas concatenadas: 17773
consumo: usando 12 meses [(2024, 7), (2024, 8)]...[(2025, 5), (2025, 6)]
  -> filas concatenadas: 1280034


## Perfilado por columna

In [13]:
def _serialize_value(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (np.datetime64, pd.Timestamp)):
        return str(pd.Timestamp(v).date())
    if isinstance(v, (np.integer, np.floating)):
        return float(v) if isinstance(v, np.floating) else int(v)
    return str(v)


def _column_all_empty(s: pd.Series) -> bool:
    """True si la columna no aporta datos (solo nulos o cadenas vacías)."""
    if len(s) == 0:
        return True
    if s.isna().all():
        return True
    if s.dtype == object or str(s.dtype) == "string":
        return s.fillna("").astype(str).str.strip().eq("").all()
    return False


def profile_dataframe(df: pd.DataFrame, source_name: str) -> dict:
    meta_cols = {"_manifest_year", "_manifest_month"}
    cols = [c for c in df.columns if c not in meta_cols]
    cols = [c for c in cols if c != "localidad1" and not _column_all_empty(df[c])]
    out = {"source": source_name, "row_count": int(len(df)), "columns": {}}
    for col in cols:
        s = df[col]
        null_pct = float(s.isna().mean()) if len(df) else 0.0
        nunique = int(s.nunique(dropna=True))
        dtype_str = str(s.dtype)
        entry = {
            "pandas_dtype": dtype_str,
            "null_fraction_observed": round(null_pct, 6),
            "nunique_observed": nunique,
        }
        # min/max numéricos o fechas
        if pd.api.types.is_numeric_dtype(s):
            smin, smax = s.min(skipna=True), s.max(skipna=True)
            entry["min_observed"] = _serialize_value(smin)
            entry["max_observed"] = _serialize_value(smax)
        elif pd.api.types.is_datetime64_any_dtype(s):
            entry["min_observed"] = _serialize_value(s.min())
            entry["max_observed"] = _serialize_value(s.max())
        # enums candidatos
        if nunique <= MAX_CARDINALITY_FOR_ENUM and nunique > 0:
            vals = sorted(s.dropna().unique().tolist(), key=lambda x: str(x))
            vals = vals[:MAX_ENUM_VALUES_LISTED]
            entry["allowed_values_observed"] = [_serialize_value(v) for v in vals]
        out["columns"][col] = entry
    return out


profiles = {}
for s, dfi in data_by_source.items():
    if dfi.empty:
        print(f"[WARN] {s}: sin datos en interim para el rango elegido.")
        profiles[s] = {"source": s, "row_count": 0, "columns": {}}
    else:
        profiles[s] = profile_dataframe(dfi, s)
        print(f"{s}: {profiles[s]['row_count']} filas, {len(profiles[s]['columns'])} columnas perfiladas")

inspecciones: 17773 filas, 11 columnas perfiladas
consumo: 1280034 filas, 16 columnas perfiladas


## Vista compacta (tabla por fuente)

In [14]:
rows = []
for s, prof in profiles.items():
    for col, info in prof.get("columns", {}).items():
        rows.append(
            {
                "source": s,
                "column": col,
                "dtype": info.get("pandas_dtype"),
                "null_frac": info.get("null_fraction_observed"),
                "nunique": info.get("nunique_observed"),
                "has_allowed_values": "allowed_values_observed" in info,
            }
        )
if rows:
    display(pd.DataFrame(rows).sort_values(["source", "column"]))
else:
    print("No hay columnas para mostrar (sin datos en interim).")

,source,column,dtype,null_frac,nunique,has_allowed_values
21,consumo,barrio,object,0.0,1253,False
16,consumo,categoria,object,0.0,6,True
14,consumo,consumo,int64,0.0,909,False
11,consumo,contrato,object,0.0,110551,False
26,consumo,date,datetime64[ns],0.0,12,True
22,consumo,determinacion_consumo,object,0.0,4,True
15,consumo,estado,object,0.0,14,True
13,consumo,fecha_mes,datetime64[ns],0.0,12,True
12,consumo,instalacion,object,0.0,154181,False
18,consumo,localidad,object,0.0,29,True


## Construir manifiesto YAML y guardar

El bloque `meta` describe cómo se generó. Las secciones bajo `sources` son las reglas **observadas** en el período; para validación productiva conviene copiar/editar y fijar `allowed_values` oficiales donde aplique.

In [15]:
manifest = {
    "meta": {
        "version": 1,
        "generated_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "generated_from": "data/interim (parquet post-ETL)",
        "lookback_months_requested": N_MONTHS,
        "max_cardinality_for_enum": MAX_CARDINALITY_FOR_ENUM,
        "note": "Borrador inferido de datos; revisar antes de usar como contrato estricto.",
    },
    "sources": profiles,
}

os.makedirs(os.path.dirname(OUTPUT_MANIFEST), exist_ok=True)
with open(OUTPUT_MANIFEST, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        manifest,
        f,
        allow_unicode=True,
        default_flow_style=False,
        sort_keys=False,
    )

print("Escrito:", OUTPUT_MANIFEST)

Escrito: d:\2024\BID\Aguas\Empresa-EPM\proyecto\ar-epm_poc\config\raw_manifest.generated.yaml


## (Opcional) Previsualizar fragmento del YAML

In [16]:
if os.path.isfile(OUTPUT_MANIFEST):
    with open(OUTPUT_MANIFEST, encoding="utf-8") as f:
        txt = f.read()
    print(txt[:4000])
    if len(txt) > 4000:
        print("\n... [truncado en vista; abrir el archivo completo en config/]")

meta:
  version: 1
  generated_at: '2026-05-28T20:18:30Z'
  generated_from: data/interim (parquet post-ETL)
  lookback_months_requested: 12
  max_cardinality_for_enum: 50
  note: Borrador inferido de datos; revisar antes de usar como contrato estricto.
sources:
  inspecciones:
    source: inspecciones
    row_count: 17773
    columns:
      solicitud:
        pandas_dtype: object
        null_fraction_observed: 0.0
        nunique_observed: 17771
      instalacion:
        pandas_dtype: int64
        null_fraction_observed: 0.0
        nunique_observed: 15250
        min_observed: 190045100000520000
        max_observed: 191481100567950016
      contrato:
        pandas_dtype: object
        null_fraction_observed: 0.0
        nunique_observed: 15939
      fecha:
        pandas_dtype: datetime64[ns]
        null_fraction_observed: 0.0
        nunique_observed: 12
        min_observed: '2024-01-01'
        max_observed: '2024-12-01'
        allowed_values_observed:
        - '1704067200